In [0]:
%run /Workspace/Users/akamil92@yahoo.com/etl/Framework/etl_logger.py

In [0]:
%run /Workspace/Users/akamil92@yahoo.com/etl/Framework/utils.py

In [0]:


from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp

# -----------------------------
# Schema
# -----------------------------
pickup_schema = StructType([
    StructField("PickupID", StringType(), True),
    StructField("ShipmentID", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("EarliestPickupTime", StringType(), True),
    StructField("LatestPickupTime", StringType(), True),
    StructField("ScheduledPickupTime", StringType(), True),
    StructField("ActualPickupTime", StringType(), True),
    StructField("EnrouteTime", StringType(), True),
    StructField("ArrivedTime", StringType(), True),
    StructField("StatusID", StringType(), True),
    StructField("LastLoadTime", TimestampType(), True)
])

table_name = "PickupLocations"
source_path = "/mnt/customer/pickup_locations.csv"
bronze_path = "/mnt/customer/bronze/PickupLocations"

try:
    df = read_safe_csv(source_path, pickup_schema).withColumn("LastLoadTime", current_timestamp())
    write_delta_safe(df, bronze_path)
    update_lastload_safe(bronze_path)
    log_step(table_name, "Bronze", "Full Load", df_after=df)
except Exception as e:
    log_step(table_name, "Bronze", "Full Load", status="FAIL", error=str(e))


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

log_path = "/mnt/customer/logs/etl_log"

# Load ETL log
df_log = spark.read.format("delta").load(log_path)

# Show the most recent 50 log entries
df_log.orderBy("Timestamp", ascending=False).show(50, truncate=False)